# Défi Quotidien : Analyse Stratégique des Performances des Supermarchés

**Notebook complet d'analyse stratégique — Veille commerciale & Visualisations interactives**

---

## 👩‍🏫 Objectifs pédagogiques
- Traduire des objectifs commerciaux en questions d'analyse exploitables
- Utiliser Matplotlib et Seaborn pour des visualisations diagnostiques
- Créer des widgets interactifs avec `ipywidgets` et `interact()`
- Extraire et présenter des insights exploitables depuis des données de vente au détail
- Structurer un carnet d'analyse professionnelle avec résumé exécutif

## 0. Imports et Configuration

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import ipywidgets as widgets
from ipywidgets import interact, Dropdown, IntSlider
from IPython.display import display
import warnings

warnings.filterwarnings('ignore')

# Style global pour des graphiques professionnels
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams.update({'figure.dpi': 100, 'axes.titlesize': 14, 'axes.labelsize': 12})

print('Imports réussis. Environnement prêt.')

## 1. Chargement et Nettoyage des Données

In [ ]:
# Chargement du dataset principal
df = pd.read_csv('superstore_dataset.csv', encoding='latin-1')

print('=== Aperçu du Dataset ===')
print(f'Dimensions : {df.shape[0]} lignes x {df.shape[1]} colonnes')
print(f'Colonnes   : {df.columns.tolist()}')
print()
df.info()
df.describe()

In [ ]:
# --- Suppression des doublons ---
n_before = len(df)
df = df.drop_duplicates()
print(f'Doublons supprimés : {n_before - len(df)}')

# --- Gestion des valeurs manquantes ---
print('Valeurs manquantes par colonne :')
missing = df.isnull().sum()
print(missing[missing > 0] if missing.sum() > 0 else 'Aucune valeur manquante.')

# Remplissage du code postal si absent
if 'Postal Code' in df.columns:
    df['Postal Code'] = df['Postal Code'].fillna(0)

# --- Conversion des colonnes de date au format datetime ---
date_columns = ['Order Date', 'Ship Date']
for col in date_columns:
    if col in df.columns:
        df[col] = pd.to_datetime(df[col])

print('\nTypes des colonnes de date après conversion :')
print(df[date_columns].dtypes)

## 2. Feature Engineering

In [ ]:
# Création des variables dérivées utiles à l'analyse stratégique
df['Profit Margin']    = (df['Profit'] / df['Sales']) * 100   # Marge bénéficiaire en %
df['Order Year']       = df['Order Date'].dt.year             # Année de commande
df['Order Month']      = df['Order Date'].dt.month            # Mois de commande
df['Order Month-Year'] = df['Order Date'].dt.to_period('M')  # Période mensuelle (ex: 2021-03)

print('Variables créées : Profit Margin, Order Year, Order Month, Order Month-Year')
df[['Sales', 'Profit', 'Profit Margin', 'Order Year', 'Order Month']].head()

## 3. Analyse Temporelle — Ventes Mensuelles Interactives

Le widget déroulant ci-dessous est **lié à la fonction `plot_monthly_sales` via `interact()`**, ce qui rend le graphique dynamique et filtrable par catégorie.

In [ ]:
# Préparation des données mensuelles agrégées par catégorie
monthly_sales = (
    df.groupby(['Order Month-Year', 'Category'])['Sales']
    .sum()
    .reset_index()
)
monthly_sales['Date'] = monthly_sales['Order Month-Year'].dt.to_timestamp()

def plot_monthly_sales(category='All'):
    """Trace l'évolution mensuelle des ventes, globale ou par catégorie sélectionnée."""
    fig, ax = plt.subplots(figsize=(14, 6))

    if category == 'All':
        # Agrégation toutes catégories confondues
        total = df.groupby('Order Month-Year')['Sales'].sum()
        ax.plot(
            total.index.to_timestamp(), total.values,
            color='steelblue', linewidth=2, marker='o', markersize=4
        )
        ax.set_title('Évolution Mensuelle des Ventes — Toutes Catégories',
                     fontsize=15, fontweight='bold')
    else:
        # Filtrage sur la catégorie choisie dans le widget
        data = monthly_sales[monthly_sales['Category'] == category]
        ax.plot(
            data['Date'], data['Sales'],
            color='darkorange', linewidth=2, marker='o', markersize=4
        )
        ax.set_title(f'Évolution Mensuelle des Ventes — Catégorie : {category}',
                     fontsize=15, fontweight='bold')

    ax.set_xlabel('Date')
    ax.set_ylabel('Ventes ($)')
    ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'${x:,.0f}'))
    ax.tick_params(axis='x', rotation=45)
    ax.grid(True, alpha=0.4)
    plt.tight_layout()
    plt.show()

# Widget déroulant : liste des catégories disponibles
category_options = ['All'] + sorted(df['Category'].unique().tolist())
dropdown_widget = widgets.Dropdown(
    options=category_options,
    value='All',
    description='Catégorie :',
    style={'description_width': 'initial'}
)

# --- Liaison du widget à la fonction via interact() — rend le graphique dynamique ---
interact(plot_monthly_sales, category=dropdown_widget);

## 4. Analyse Géographique — Top N États par Ventes (Interactif)

Graphique **horizontal (`barh`)** avec étiquettes de valeur et **filtrage dynamique Top N** via un slider `IntSlider` lié à `interact()`.

In [ ]:
# Agrégation des ventes et profits par État
state_stats = (
    df.groupby('State')
    .agg(Sales=('Sales', 'sum'), Profit=('Profit', 'sum'))
    .sort_values('Sales')
)

def plot_top_states(top_n=10):
    """Trace un graphique barh des Top N états avec étiquettes de valeur et couleur profit/perte."""
    from matplotlib.patches import Patch

    # Sélection des N états avec les ventes les plus élevées
    top = state_stats.tail(top_n)

    # Couleur verte si profit positif, rouge sinon
    colors = ['#d73027' if p < 0 else '#1a9850' for p in top['Profit']]

    fig, ax = plt.subplots(figsize=(12, max(5, top_n * 0.5)))

    # Graphique horizontal (barh) — tel que spécifié dans les consignes
    bars = ax.barh(top.index, top['Sales'], color=colors, edgecolor='white', height=0.7)

    # Étiquettes de valeur sur chaque barre
    for bar, val in zip(bars, top['Sales']):
        ax.text(
            bar.get_width() + top['Sales'].max() * 0.01,
            bar.get_y() + bar.get_height() / 2,
            f'${val:,.0f}',
            va='center', ha='left', fontsize=9
        )

    ax.set_title(
        f'Top {top_n} États par Volume de Ventes — Analyse des Performances Géographiques',
        fontsize=13, fontweight='bold'
    )
    ax.set_xlabel('Ventes Totales ($)')
    ax.set_ylabel('État')
    ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'${x/1e3:.0f}K'))

    # Légende couleur profit/perte
    legend_elements = [
        Patch(facecolor='#1a9850', label='Profit positif'),
        Patch(facecolor='#d73027', label='Profit négatif')
    ]
    ax.legend(handles=legend_elements, loc='lower right')

    plt.tight_layout()
    plt.show()

# Slider Top N
slider_widget = widgets.IntSlider(
    min=5, max=25, step=1, value=10,
    description='Top N états :',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='400px')
)

# --- Liaison du slider à la fonction via interact() — filtrage dynamique Top N ---
interact(plot_top_states, top_n=slider_widget);

## 5. Top 10 des Produits les Plus Rentables — Analyse des Performances Produits

Graphique Seaborn avec **annotations de valeur** sur chaque barre.

In [ ]:
# Agrégation du profit total par produit et sélection du Top 10
product_profit = (
    df.groupby('Product Name')['Profit']
    .sum()
    .sort_values(ascending=False)
    .head(10)
)

fig, ax = plt.subplots(figsize=(13, 7))

# Graphique Seaborn
sns.barplot(
    x=product_profit.values,
    y=product_profit.index,
    palette='YlOrRd_r',
    ax=ax
)

# Annotations de valeur sur chaque barre (exigence clé)
for bar, val in zip(ax.patches, product_profit.values):
    ax.text(
        bar.get_width() + product_profit.max() * 0.01,
        bar.get_y() + bar.get_height() / 2,
        f'${val:,.0f}',
        va='center', ha='left', fontsize=10, fontweight='bold'
    )

ax.set_title(
    'Top 10 des Produits les Plus Rentables — Résumé — Analyse des Performances Produits',
    fontsize=13, fontweight='bold'
)
ax.set_xlabel('Profit Total ($)')
ax.set_ylabel('Produit')
ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'${x:,.0f}'))
plt.tight_layout()
plt.show()

## 6. Remise vs Profit — Impact de la Stratégie de Remises

Nuage de points coloré par catégorie avec **courbe de tendance `regplot()`** superposée.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 7))

# Nuage de points par catégorie
categories = df['Category'].unique()
palette = sns.color_palette('Set1', len(categories))
for cat, color in zip(categories, palette):
    subset = df[df['Category'] == cat]
    ax.scatter(
        subset['Discount'], subset['Profit'],
        color=color, alpha=0.35, s=22, label=cat
    )

# Courbe de tendance globale superposée via regplot() — intervalle de confiance à 95 %
sns.regplot(
    data=df,
    x='Discount',
    y='Profit',
    scatter=False,          # Le nuage est déjà tracé ci-dessus
    color='black',
    line_kws={'linewidth': 2, 'linestyle': '--', 'label': 'Tendance globale'},
    ci=95,
    ax=ax
)

# Ligne de référence seuil de rentabilité
ax.axhline(0, color='red', linewidth=1.2, linestyle=':', label='Seuil profit = 0')

ax.set_title(
    'Remise vs Profit — Impact de la Stratégie de Remises sur la Rentabilité',
    fontsize=13, fontweight='bold'
)
ax.set_xlabel('Remise (Discount)')
ax.set_ylabel('Profit ($)')
ax.legend(title='Catégorie')
plt.tight_layout()
plt.show()

# Indicateur de corrélation quantitative
corr = df['Discount'].corr(df['Profit'])
print(f'Corrélation Remise–Profit : {corr:.4f} (relation négative confirmée)')

## 7. Revue de la Méthodologie et des Outils — Matplotlib vs Seaborn

| Critère | **Matplotlib** | **Seaborn** |
|---|---|---|
| Niveau d'abstraction | Bas niveau — contrôle total | Haut niveau — syntaxe concise |
| Facilité d'utilisation | Verbeux, configuration manuelle | Rapide pour les graphiques statistiques |
| Graphiques statistiques | Code supplémentaire nécessaire | Natifs (regplot, boxplot, heatmap…) |
| Personnalisation | Maximale (chaque élément configurable) | Limitée mais suffisante |
| Intégration pandas | Indirecte (valeurs numpy) | Directe (paramètre `data=df`) |
| Interactivité | Via ipywidgets | Via ipywidgets |
| Cas d'usage idéal | Graphiques sur mesure, publications | Exploration rapide, analyses statistiques |

### Synthèse méthodologique
Dans ce projet, les deux librairies ont été utilisées de façon **complémentaire** :
- **Seaborn** pour les graphiques d'exploration tirant parti de ses intégrations statistiques natives (`barplot`, `scatterplot`, `regplot`).
- **Matplotlib** pour les personnalisations avancées : annotations de valeur sur les barres, couleurs conditionnelles (profit/perte), mise en page fine.

Cette combinaison offre le meilleur équilibre entre productivité et maîtrise du rendu visuel.

## 8. Résumé Exécutif — Analyse Stratégique des Performances du Supermarché

In [ ]:
# === Calcul des indicateurs clés de performance ===

# KPIs globaux
total_sales   = df['Sales'].sum()
total_profit  = df['Profit'].sum()
global_margin = (total_profit / total_sales) * 100

# Performances par catégorie
cat_perf = (
    df.groupby('Category')
    .agg(Sales=('Sales', 'sum'), Profit=('Profit', 'sum'))
    .assign(Margin=lambda x: (x['Profit'] / x['Sales']) * 100)
)

# Performances géographiques
state_profit = df.groupby('State')['Profit'].sum()
top_state    = state_profit.idxmax()
flop_state   = state_profit.idxmin()

# Impact des remises élevées (> 20 %)
high_disc   = df[df['Discount'] > 0.20]
pct_loss_hd = (high_disc['Profit'] < 0).mean() * 100
corr_disc   = df['Discount'].corr(df['Profit'])

# === Affichage du résumé structuré ===
print('=' * 65)
print('   RÉSUMÉ EXÉCUTIF — ANALYSE STRATÉGIQUE DU SUPERMARCHÉ')
print('=' * 65)

print(f"""
PERFORMANCES GLOBALES
  Ventes totales     : ${total_sales:>12,.2f}
  Profit total       : ${total_profit:>12,.2f}
  Marge bénéficiaire : {global_margin:>11.2f} %

PERFORMANCES PAR CATÉGORIE""")
for cat, row in cat_perf.iterrows():
    print(f'  {cat:<26} Ventes: ${row["Sales"]:>10,.0f}  '
          f'Profit: ${row["Profit"]:>9,.0f}  Marge: {row["Margin"]:5.1f}%')

print(f"""
PERFORMANCES GÉOGRAPHIQUES
  État le plus rentable  : {top_state} (${state_profit[top_state]:,.0f})
  État le moins rentable : {flop_state} (${state_profit[flop_state]:,.0f})
  Observation : un volume de ventes élevé ne garantit pas
  la rentabilité — une analyse coût/marge par région s'impose.

STRATÉGIE DE REMISES
  Corrélation Remise–Profit     : {corr_disc:.4f} (relation négative)
  Ventes déficitaires (Remise > 20%) : {pct_loss_hd:.1f} %
  Observation : au-delà de 20 % de remise, {pct_loss_hd:.0f} % des ventes
  sont déficitaires. Un plafonnement à 15-20 % est recommandé.

RECOMMANDATIONS STRATÉGIQUES
  1. Renforcer les catégories à marge élevée et réduire
     les promotions excessives sur les segments déficitaires.
  2. Cibler les ressources commerciales sur les états à fort
     potentiel et revoir la politique tarifaire dans les zones
     en perte chronique.
  3. Plafonner les remises à 20 % maximum pour préserver
     la rentabilité globale.
""")
print('=' * 65)